# First-Pass Batch 4

This notebook handles the deterministic first pass for batch 4 of 4. It does not retry failed rows.


## Drive And Repo Setup

Mount Drive, clone the repo into the runtime, and copy the required CSV inputs into the checkout.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output root
- Rerun-safe: Yes. It reclones the repo and recopies the inputs cleanly.


In [6]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "svg-kaggle-comptetition" / "submission_outputs"

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    if destination.exists():
        continue

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    copied = False
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            copied = True
            break
    if not copied:
        for candidate in DRIVE_ROOT.rglob(required_name):
            if candidate.is_file():
                shutil.copy2(candidate, destination)
                copied = True
                break
    if not copied:
        raise FileNotFoundError(
            f"Could not find {required_name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
        )

os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)
sys.path.insert(0, str(CHECKOUT_PATH))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Drive output root: {OUTPUT_ROOT}")


Mounted at /content/drive
Repo checkout: /content/svg-kaggle-comptetition
Drive output root: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs


## Package Check

Install any missing runtime packages required by the shared helper module and this notebook.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs missing packages.


In [4]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "cairosvg": "cairosvg",
    "skimage": "scikit-image",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


All required packages are already installed.


## Load Helpers, Recommendation, And Batch Slice

Import the shared helper module, load the analysis recommendation, and select this notebook's fixed quarter of `test.csv`.

- Reads: Repo checkout, helper module, analysis recommendation, competition CSV files
- Writes: In-memory batch slice only
- Rerun-safe: Yes. It is deterministic because the batch boundaries are fixed.


In [7]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

import submission_pipeline as pipeline

pipeline.set_global_seed(1337)
PATHS = pipeline.resolve_pipeline_paths(Path(os.environ["SVG_KAGGLE_REPO_ROOT"]), OUTPUT_ROOT)
CONFIG = pipeline.GenerationConfig(verbose_progress=True)
TEST_DF, SAMPLE_SUBMISSION_DF, TRAIN_DF = pipeline.load_competition_frames(PATHS)

print(f"Repo root: {PATHS.repo_root}")
print(f"Output root: {PATHS.output_root}")

BATCH_ID = 4
ANALYSIS_RECOMMENDATION = pipeline.load_analysis_recommendation(PATHS)
GENERATION_MODE = ANALYSIS_RECOMMENDATION.get("recommended_generation_mode", "body_only")
BATCH_DF = pipeline.select_batch_df(TEST_DF, batch_id=BATCH_ID, batch_count=4)
BATCH_DIR = pipeline.batch_output_dir(PATHS, BATCH_ID)
print(f"Recommended generation mode: {GENERATION_MODE}")
print(f"Batch rows: {len(BATCH_DF)}")
display(BATCH_DF[["id", "prompt", "row_index"]].head())


Repo root: /content/svg-kaggle-comptetition
Output root: /content/drive/MyDrive/svg-kaggle-comptetition/submission_outputs
Recommended generation mode: full_svg
Batch rows: 250


,id,prompt,row_index
0,66eec18f9c8a1b693aab52c1eb617116,The image shows a dark blue rectangular block ...,750
1,1746e618853fd17448f2740f90622e42,A slice of cheese with a rind is depicted in a...,751
2,af4a0c3d71c82e28ce9d41f9ee8c4c2d,The image contains a solid black color occupyi...,752
3,df01fd16ec344d444235519b71c8c738,An orange rectangular object with three horizo...,753
4,6cd2ff67e8c40d994115dc735aab04ab,A black speech bubble icon with two white hori...,754


## Load Model

Load the base model plus the LoRA adapter into the current Colab GPU runtime for first-pass generation.

- Reads: Base model on Hugging Face, LoRA adapter from repo
- Writes: In-memory model state only
- Rerun-safe: No. It is safe to rerun, but it reloads the model and costs time.


In [8]:
RUNTIME = pipeline.load_lora_runtime(PATHS.adapter_dir)
print(f"Loaded runtime on: {RUNTIME.runtime_device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded runtime on: cuda


## Deterministic First Pass

Run one deterministic generation attempt per row in this batch and save the success/failure artifacts for the merge notebook.

- Reads: Selected batch rows, model runtime
- Writes: Batch output CSV artifacts under this batch directory
- Rerun-safe: Yes. It recomputes and overwrites this batch's artifacts.


In [9]:
FIRST_PASS_RESULTS_DF = pipeline.run_first_pass_batch(
    BATCH_DF,
    RUNTIME,
    batch_id=BATCH_ID,
    generation_mode=GENERATION_MODE,
    config=CONFIG,
)
pipeline.save_batch_outputs(FIRST_PASS_RESULTS_DF, BATCH_DIR)
print(f"Wrote batch artifacts to: {BATCH_DIR}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[batch_4 1/250] row 66eec18f9c8a1b693aab52c1eb617116: attempt 1/1 (deterministic)
[batch_4 1/250] row 66eec18f9c8a1b693aab52c1eb617116: success on attempt 1/1
[batch_4 2/250] row 1746e618853fd17448f2740f90622e42: attempt 1/1 (deterministic)
[batch_4 2/250] row 1746e618853fd17448f2740f90622e42: attempt 1/1 failed: unclosed token: line 3, column 0
[batch_4 3/250] row af4a0c3d71c82e28ce9d41f9ee8c4c2d: attempt 1/1 (deterministic)
[batch_4 3/250] row af4a0c3d71c82e28ce9d41f9ee8c4c2d: success on attempt 1/1
[batch_4 4/250] row df01fd16ec344d444235519b71c8c738: attempt 1/1 (deterministic)
[batch_4 4/250] row df01fd16ec344d444235519b71c8c738: success on attempt 1/1
[batch_4 5/250] row 6cd2ff67e8c40d994115dc735aab04ab: attempt 1/1 (deterministic)
[batch_4 5/250] row 6cd2ff67e8c40d994115dc735aab04ab: success on attempt 1/1
[batch_4 6/250] row 9ad34e074349bc67029395238ce85ab5: attempt 1/1 (deterministic)
[batch_4 6/250] row 9ad34e074349bc67029395238ce85ab5: attempt 1/1 failed: unclosed token: lin

## Batch Summary

Show the first-pass success/failure totals and inspect a few failed rows with their gate errors.

- Reads: Batch result artifacts in memory
- Writes: Nothing
- Rerun-safe: Yes. Read-only display cell.


In [10]:
display(pipeline.summarize_batch_results(FIRST_PASS_RESULTS_DF))
FAILED_DF = FIRST_PASS_RESULTS_DF[~FIRST_PASS_RESULTS_DF["first_pass_success"]].copy()
if FAILED_DF.empty:
    print("No failed rows in this batch.")
else:
    display(FAILED_DF[["id", "row_index", "generation_mode", "gate_error"]].head(20))


,metric,value
0,rows,250.000000
1,first_pass_successes,120.000000
2,first_pass_failures,130.000000
3,mean_runtime_seconds,14.696688
4,mean_char_len_successes,350.075000


,id,row_index,generation_mode,gate_error
1,1746e618853fd17448f2740f90622e42,751,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
5,9ad34e074349bc67029395238ce85ab5,755,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
6,46a4faf1ae6a4426e6e96a35abb5074f,756,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
7,11c1b206357e957e417b6fb820eb7ad0,757,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
8,b49e57348a42e770f507d12b21652c1f,758,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
10,ede87c9406eefbb3f9a689b9f30201c6,760,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
17,0a91f876f6c2294c12c2213f23830a4f,767,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
18,fa5c275c8505741224c926183f9170d7,768,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
19,d3dd2646d2862a60cedd6f8ce7112a34,769,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
22,4f0f1ca31425d5e73907dd50ad5ea209,772,full_svg,Failed after 1 attempts. attempt 1: unclosed t...
